In [21]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
#using PyPlot
#using PyCall
using BenchmarkTools
using Random
#hp = pyimport("healpy")
include("../src/function2/rotator.jl")
include("../src/function2/tools.jl")

unique_theta_val (generic function with 1 method)

# Two time rotation

$$
D^{\ell}_{mn}(\phi + \delta \phi, \theta + \delta \theta, \psi + \delta \psi)
= \sum_{M} D^{\ell}_{mM}(\phi, \theta, \psi) D^{\ell}_{Mn}(\alpha, \beta, \gamma)
$$

$$

R(\phi,\theta,\psi)=R_z(\phi)R_y(\theta)R_z(\psi),\qquad\\
R_2 = R(\phi,\theta,\psi)^{-1}\,R(\phi+\delta\phi,\theta+\delta\theta,\psi+\delta\psi) \\
\beta = \arccos\!\left((R_2)_{33}\right),\qquad \\
\alpha = \operatorname{atan2}\!\left((R_2)_{23},(R_2)_{13}\right),\qquad \\

\gamma = \operatorname{atan2}\!\left((R_2)_{32},-(R_2)_{31}\right)

$$

In [2]:
ell =  700
θ = pi/3
φ = pi/4
dθ = 1e-3
dφ = -1e-3
ψ = pi/6
err, (α2, β2, γ2) = check_split(φ, θ, dφ, dθ, ψ)

(5.911937606128959e-14, (-0.7141170019360827, 0.0013230392084807063, 1.2372162105158164))

In [3]:
D_tar = WignerD.wignerD(ell, φ + dφ, θ + dθ, ψ);

In [4]:
D_1st = WignerD.wignerD(ell, φ, θ, 0.0);

In [5]:
D_2nd = WignerD.wignerD(ell, α2, β2, γ2);

In [6]:
D_2time = D_1st * D_2nd ;

In [7]:
maximum(abs.(D_tar .- D_2time))

6.830101375729925e-12

# All-sky convolution
## How to calculate WignerD matrices

From All-sky convolution
There is algorithm to calculate WignerD

$$
D_{mn}^{\ell}(\phi, \theta, \psi) = \sum_{|M| \leq \ell} \left[ D^{\ell}_{mM}(\phi - \pi/2, -\pi/2, \theta)\times D^{\ell}_{Mn}(0, \pi/2, \psi + \pi/2) \right] \\
= \sum_{|M| \leq \ell} \left[ \exp(-im(\phi - \pi/2)) d^{\ell}_{mM}(-\pi/2)\exp(-iM\theta)\times d^{\ell}_{Mn}(\pi/2)\exp(-in(\psi + \pi/2)) \right] \\
= \sum_{|M| \leq \ell} \left[ \exp(-im(\phi - \pi/2)) d^{\ell}_{Mm}(\pi/2)\exp(-iM\theta)\times d^{\ell}_{Mn}(\pi/2)\exp(-in(\psi + \pi/2)) \right]
$$
convinient equation
$$
d^{\ell}_{mn}\left(\frac{\pi}{2}\right) = (-1)^{\ell-n}d_{-mn}^{\ell}\left(\frac{\pi}{2}\right)=(-1)^{\ell+m}d^{\ell}_{m-n}\left(\frac{\pi}{2}\right)=(-1)^{m-n}d_{-m-n}^{\ell}\left(\frac{\pi}{2}\right) \\
d_{m-n}^{\ell}\left( \frac{\pi}{2}\right) = (-1)^{\ell+m}d_{mn}^{\ell} \left( \frac{\pi}{2}\right) 
$$

# All-sky convolution
## How to calculate WignerD matrices

From All-sky convolution
There is algorithm to calculate WignerD

$$
d_{mn}^{\ell}(\theta) = D_{mn}^{\ell}(0, \theta, 0) = \sum_{|M| \leq \ell} \left[ D^{\ell}_{mM}(- \pi/2, -\pi/2, \theta)\times D^{\ell}_{Mn}(0, \pi/2,  \pi/2) \right] \\
= \sum_{|M| \leq \ell} \left[ \exp(-im(- \pi/2)) d^{\ell}_{mM}(-\pi/2)\exp(-iM\theta)\times d^{\ell}_{Mn}(\pi/2)\exp(-in( \pi/2)) \right] \\
= \sum_{|M| \leq \ell} \left[  d^{\ell}_{Mm}(\pi/2)\exp(-iM\theta)\exp(im(\pi/2)) \times d^{\ell}_{Mn}(\pi/2)\exp(-in(\pi/2)) \right]
$$
convinient equation
$$
d^{\ell}_{mn}\left(\frac{\pi}{2}\right) = (-1)^{\ell-n}d_{-mn}^{\ell}\left(\frac{\pi}{2}\right)=(-1)^{\ell+m}d^{\ell}_{m-n}\left(\frac{\pi}{2}\right)=(-1)^{m-n}d_{-m-n}^{\ell}\left(\frac{\pi}{2}\right) \\
d_{m-n}^{\ell}\left( \frac{\pi}{2}\right) = (-1)^{\ell+m}d_{mn}^{\ell} \left( \frac{\pi}{2}\right) \\
d^{\ell}_{mn}\left(-\theta \right) = d^{\ell}_{nm}\left( \theta \right)
$$

In [14]:
ell, phi, theta, psi = 100, pi/3, pi/4, pi/6

(100, 1.0471975511965976, 0.7853981633974483, 0.5235987755982988)

In [15]:
d = @btime WignerD.wignerd(ell,pi/2);

  51.776 ms (4 allocations: 947.09 KiB)


In [16]:
phi, theta, psi = pi/3, pi/1, pi/6
D = @btime  WignerD.wignerD(ell, phi, theta, psi)
Dp = @btime  WignerD_calculator_fast(d, ell, phi, theta, psi);

  53.206 ms (4 allocations: 1.23 MiB)
  5.458 ms (8 allocations: 1.24 MiB)


In [18]:
nside =128

128

In [22]:
unique_theta_val(nside)

511-element Vector{Float64}:
 0.006378890353433737
 0.012757845597670932
 0.019136930629456345
 0.02551621035741883
 0.0318957497080172
 0.038275613631490575
 0.04465586710781478
 0.05103657515266638
 0.05741780282339575
 0.06379961522501076
 0.07018207751617268
 0.07656525491520565
 0.08294921270612156
 ⋮
 3.0650273986745877
 3.0714105760736206
 3.0777930383647827
 3.0841748507663973
 3.090556078437127
 3.0969367864819786
 3.1033170399583025
 3.109696903881776
 3.1160764432323744
 3.122455722960337
 3.1288348079921224
 3.1352137632363597

In [23]:
0.006378890353433737 + 3.1352137632363597

3.1415926535897936

In [24]:
0.012757845597670932 + 3.1288348079921224

3.1415926535897936

In [127]:
nside = 4
npix = nside2npix(nside)
res = Resolution(nside)

Healpix resolution(NSIDE = 4)

In [129]:
num = 5000
Random.seed!(10)
phi = rand(num)*2pi
theta = rand(num)*pi
psi = rand(num)*2pi;

In [131]:
pix = zeros(Int64, 5000);

In [151]:
for phi_loop in enumerate(phi)
    idx = phi_loop[1]
    pix[idx] = ang2pixRing(res, theta[idx] ,phi[idx])
end

In [147]:
ang2pixRing(res, theta[1] ,phi[1])

192

In [153]:
pix

5000-element Vector{Int64}:
 192
  36
 191
 125
  58
   6
 131
 189
   4
 164
  69
  76
   2
   ⋮
  73
  18
 172
   2
 158
 191
 189
 137
  16
 189
   1
  66